## Data Collection and Loading

In [121]:
import pandas as pd
import numpy as np

In [122]:
df = pd.read_csv('../data/interim/customer_churn_clean.csv')

## Basic Data Inspection & Overview

In [123]:
df.head(10)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,Female,0,No,No,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes
6,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.40,No
7,Female,0,No,No,10,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.90,No
8,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
9,Male,0,No,Yes,62,Yes,No,DSL,Yes,Yes,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


In [124]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   str    
 1   SeniorCitizen     7032 non-null   int64  
 2   Partner           7032 non-null   str    
 3   Dependents        7032 non-null   str    
 4   tenure            7032 non-null   int64  
 5   PhoneService      7032 non-null   str    
 6   MultipleLines     7032 non-null   str    
 7   InternetService   7032 non-null   str    
 8   OnlineSecurity    7032 non-null   str    
 9   OnlineBackup      7032 non-null   str    
 10  DeviceProtection  7032 non-null   str    
 11  TechSupport       7032 non-null   str    
 12  StreamingTV       7032 non-null   str    
 13  StreamingMovies   7032 non-null   str    
 14  Contract          7032 non-null   str    
 15  PaperlessBilling  7032 non-null   str    
 16  PaymentMethod     7032 non-null   str    
 17  Monthl

## Build Features

In [125]:
# จัดกลุ่มระยะเวลาการเป็นลูกค้า (tenure) ออกเป็น 4 ช่วง
# เพื่อช่วยให้โมเดลเรียนรู้พฤติกรรมของลูกค้าในแต่ละช่วงอายุการใช้งานได้ง่ายขึ้น
# เช่น ลูกค้าใหม่ (0-12 เดือน) อาจมีโอกาส Churn สูงกว่าลูกค้าที่ใช้งานมานาน
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-12', '13-24', '25-48', '49-72']
)

In [126]:
# สร้าง Feature เพื่อระบุว่าลูกค้าเป็นสัญญาระยะยาวหรือไม่
# โดยลูกค้าที่มีสัญญา One year หรือ Two year จะถูกกำหนดเป็น 1
# และลูกค้าที่ใช้สัญญา Month-to-month จะถูกกำหนดเป็น 0
# เนื่องจากลูกค้าสัญญาระยะยาวมักมีแนวโน้ม Churn ต่ำกว่า
df['long_term_contract'] = (
    df['Contract'] != 'Month-to-month'
).astype(int)

In [127]:
# สร้าง Feature เพื่อระบุว่าลูกค้าใช้บริการอินเทอร์เน็ตแบบ Fiber optic หรือไม่
# กำหนดค่าเป็น 1 หากใช้ Fiber optic และ 0 สำหรับประเภทอื่น
# เนื่องจากการวิเคราะห์เบื้องต้นพบว่ากลุ่มลูกค้า Fiber optic
# มีอัตราการ Churn สูงกว่ากลุ่ม DSL และ No internet service
df['is_fiber'] = (
    df['InternetService'] == 'Fiber optic'
).astype(int)

In [128]:
# สร้าง Feature เพื่อระบุว่าลูกค้ามีบริการ TechSupport หรือไม่
# กำหนดค่าเป็น 1 หากมีบริการ TechSupport และ 0 หากไม่มี
# เนื่องจากลูกค้าที่ได้รับการสนับสนุนทางเทคนิคมักมีความพึงพอใจสูงกว่า
# และมีแนวโน้ม Churn ต่ำกว่าลูกค้าที่ไม่มีบริการดังกล่าว
df['has_tech_support'] = (
    df['TechSupport'] == 'Yes'
).astype(int)

In [129]:
# สร้าง Feature เพื่อระบุว่าลูกค้าใช้ช่องทางชำระเงินแบบ Electronic check หรือไม่
# กำหนดค่าเป็น 1 หากใช้ Electronic check และ 0 สำหรับช่องทางอื่น
# เนื่องจากการวิเคราะห์ข้อมูลพบว่าลูกค้ากลุ่มนี้มีแนวโน้ม Churn สูงกว่ากลุ่มที่ใช้วิธีชำระเงินประเภทอื่น
df['electronic_payment'] = (
    df['PaymentMethod'] == 'Electronic check'
).astype(int)

In [130]:
# สร้าง Feature เพื่อระบุลูกค้าใหม่ที่มีค่าใช้จ่ายรายเดือนสูง
# กำหนดค่าเป็น 1 หากเป็นลูกค้าที่ใช้งานไม่เกิน 12 เดือน
# และมี MonthlyCharges สูงกว่าค่ามัธยฐานของข้อมูลทั้งหมด
# เนื่องจากลูกค้ากลุ่มนี้อาจมีความอ่อนไหวต่อราคา
# และมีแนวโน้ม Churn สูงกว่ากลุ่มอื่น
df['high_charge_new_customer'] = (
    (df['tenure'] < 12) &
    (df['MonthlyCharges'] > df['MonthlyCharges'].median())
).astype(int)

In [131]:
# สร้าง Feature เพื่อระบุลูกค้าที่ใช้ Fiber optic แต่ไม่มี TechSupport
# กำหนดค่าเป็น 1 หากใช้ Fiber optic และไม่มีบริการ TechSupport
# และกำหนดค่าเป็น 0 สำหรับกรณีอื่น
# เนื่องจากการวิเคราะห์พบว่าลูกค้ากลุ่มนี้อาจมีความเสี่ยง Churn สูง
# จากการใช้งานบริการที่มีความซับซ้อนโดยไม่ได้รับการสนับสนุนทางเทคนิค
df['fiber_no_support'] = (
    (df['InternetService'] == 'Fiber optic') &
    (df['TechSupport'] == 'No')
).astype(int)

In [132]:
# สร้าง Feature เพื่อระบุลูกค้าที่ใช้ Fiber optic แต่ไม่มี OnlineSecurity
# กำหนดค่าเป็น 1 หากใช้ Fiber optic และไม่มีบริการ OnlineSecurity
# และกำหนดค่าเป็น 0 สำหรับกรณีอื่น
# เนื่องจากลูกค้ากลุ่มนี้อาจมีความเสี่ยง Churn สูง
# จากการใช้บริการอินเทอร์เน็ตความเร็วสูงโดยไม่มีบริการด้านความปลอดภัยเพิ่มเติม
df['fiber_no_security'] = (
    (df['InternetService'] == 'Fiber optic') &
    (df['OnlineSecurity'] == 'No')
).astype(int)

In [133]:
# สร้าง Feature เพื่อระบุลูกค้าที่มีความเสี่ยง Churn สูง
# กำหนดค่าเป็น 1 หากใช้สัญญาแบบ Month-to-month
# และชำระเงินผ่าน Electronic check
# และกำหนดค่าเป็น 0 สำหรับกรณีอื่น
# เนื่องจากผลการวิเคราะห์พบว่าทั้งสองปัจจัยมีความสัมพันธ์กับการ Churn สูง
df['high_risk_customer'] = (
    (df['Contract'] == 'Month-to-month') &
    (df['PaymentMethod'] == 'Electronic check')
).astype(int)

## Encoding & Scaling